# 08 — Process domestic demand

Build annual domestic diesel/gasoline demand from JODI demand observations and leave an explicit DGEG cross-check hook.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.jodi import read_secondary_zip, canonicalise_secondary, filter_portugal_fuels, annualise
from portugal_refining_resilience.validation import assert_nonnegative, assert_unique


In [ ]:
raw_zip = PATHS.raw / "jodi" / "world_secondary_csv.zip"
if not raw_zip.exists():
    raise FileNotFoundError("Run notebook 03 first. Domestic demand has no seed fallback because it should not be fabricated.")
raw = read_secondary_zip(raw_zip)
canonical = canonicalise_secondary(raw)
selected = filter_portugal_fuels(canonical, flows=("demand",))
annual = annualise(selected).rename(columns={"product_canonical": "product", "value_kt": "demand_kt"})
annual = annual.loc[annual["year"].between(2005, 2024), ["year", "product", "demand_kt", "source"]]
assert_nonnegative(annual, ["demand_kt"])
assert_unique(annual, ["year", "product"])
persist_dataframe(annual, PATHS.processed / "fuel_demand_annual.csv", key_columns=["year", "product"], metadata={"unit": "kt"})
display(annual.tail())


### DGEG cross-check

Once the DGEG long sales workbook has been inspected, add a source-specific extraction into `data/interim/` and compare the road-fuel sales series with JODI demand. Do not overwrite disagreements; persist both and explain the statistical concept difference.
